# YOLOFUTBOL: curva de presión a partir de un vídeo de fútbol

Este cuaderno analiza un clip de fútbol grabado en plano general y señala al entrenador **dónde mirar**: los momentos en que más rivales aprietan al jugador con balón.

Al terminar tendrás:

- `pressure.png`: la curva de presión (rivales presionando a lo largo del clip) con los momentos de presión máxima marcados.
- `annotated.mp4`: el vídeo con los equipos, el poseedor, los rivales que presionan y una vista cenital del campo.
- `sequences.csv`: una fila por posesión, con cuándo llega el primer presionante, cuántos rivales presionan de media y cómo termina.

**Antes de empezar:** en el menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución* elige **GPU (T4)**. Sin GPU funciona, pero tarda al menos 10 veces más.

Ejecuta las celdas en orden con el botón ▶ de cada una.

## 1. Instalación

Descarga el programa, instala las librerías y descarga los modelos de detección y un clip de ejemplo. Tarda unos minutos.

In [ ]:
!git clone https://github.com/javiervales/yolofutbol.git
%cd yolofutbol
# PyTorch ya viene instalado en Colab; se instala el resto.
!pip install -q -r requirements.txt gdown
!bash descargar_modelos.sh --clip-ejemplo

## 2. Elegir el vídeo

Ejecuta la celda y, si quieres analizar un clip propio, súbelo con el botón *Elegir archivos* (mp4, plano general, a ser posible 1080p). Si cancelas la subida o no eliges nada, se usa el clip de ejemplo `data/0bfacc_0.mp4`.

In [ ]:
import os, shutil
from google.colab import files

CLIP = "data/0bfacc_0.mp4"   # clip de ejemplo
try:
    subidos = files.upload()
except Exception:
    subidos = {}
if subidos:
    nombre = list(subidos)[0]
    shutil.move(nombre, os.path.join("data", nombre))
    CLIP = os.path.join("data", nombre)
print("Se analizará:", CLIP)

## 3. Analizar

Un solo comando con los parámetros del artículo. Para un clip de 30 s con GPU tarda unos minutos. Al final imprime un resumen de las posesiones.

In [ ]:
!bash analizar.sh "$CLIP"

## 4. Ver los resultados

La curva: puntos grises, rivales presionando en cada fotograma; línea de color, su media móvil; triángulos, momentos de presión máxima. Donde no hay línea, el sistema no ha asignado la posesión a ningún equipo.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

SALIDA = Path("salidas") / Path(CLIP).stem
display(Image(filename=str(SALIDA / "pressure.png")))

pos = pd.read_csv(SALIDA / "sequences.csv")
tabla = pd.DataFrame({
    "equipo": pos["team"].map({0: "azul", 1: "naranja"}),
    "inicio (s)": pos["start_t_s"].round(1),
    "duración (s)": pos["duration_s"].round(1),
    "primer presionante (s)": pos["time_to_first_presser_s"].round(1),
    "presionantes medios": pos["mean_pressers"].round(2),
    "presionantes máx.": pos["max_pressers"],
    "toques": pos["n_touches"],
    "termina": pos["end_reason"].map({"lost_to_opponent": "la recupera el rival",
                                      "ball_lost": "balón perdido de vista",
                                      "clip_end": "fin del clip"}),
})
tabla

## 5. Descargar todo

Comprime la carpeta de resultados (curva, vídeo anotado y tablas) y la descarga a tu ordenador.

In [ ]:
!cd salidas && zip -qr ../resultados_yolofutbol.zip .
files.download("resultados_yolofutbol.zip")